In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

In [3]:
CSV_PATH = r"C:\Users\Admin\Desktop\TANPHAT\Dự án_canhan\TIKTOK_V2\clustering\creator_features_final4.csv"
df = pd.read_csv(CSV_PATH)
print(f"Raw data: {df.shape[0]} rows, {df.shape[1]} cols")

Raw data: 2849 rows, 60 cols


In [4]:
print(df.shape)
print(df.columns.tolist())
df.describe(percentiles=[0.5, 0.9, 0.99]).T

(2849, 60)
['CREATOR_ID', 'VIDEO_ID', 'CREATE_TIME', 'FOLLOWERS', 'FOLLOWING_COUNT', 'ENGAGEMENT', 'TOTAL_LIKES', 'DIGG_COUNT', 'VIDEO_COUNT', 'COLLAB_SCORE', 'VIEW_COUNT', 'LIKE_COUNT', 'COMMENT_COUNT', 'SHARE_COUNT', 'SAVE_COUNT', 'VQSCORE', 'BITRATE', 'CATEGORY_TYPE', 'CATEGORY_y', 'PRICE_NUM', 'HAS_SHOP_LINK', 'hour', 'CREATOR_TIER', 'DAY_OF_WEEK', 'MEAN_LIKES_3M', 'P90_LIKES_3M', 'MAX_LIKES_3M', 'MEAN_COMMENTS_3M', 'P90_COMMENTS_3M', 'MAX_COMMENTS_3M', 'MEAN_SHARES_3M', 'P90_SHARES_3M', 'MAX_SHARES_3M', 'MEAN_SAVES_3M', 'P90_SAVES_3M', 'MAX_SAVES_3M', 'MEAN_VIEWS_3M', 'P90_VIEWS_3M', 'MAX_VIEWS_3M', 'VIDEO_COUNT_3M_x', 'MEAN_VIEWS_RATE_3M', 'MEAN_LIKE_RATE_3M', 'MEAN_COMMENT_RATE_3M', 'MEAN_SHARE_RATE_3M', 'MEAN_SAVE_RATE_3M', 'VIDEO_COUNT_3M_y', 'unique_days', 'AVG_POST_GAP', 'VIDEOS_PER_WEEK', 'POSTING_CONSISTENCY', 'VIEW_MEAN', 'VIEW_STD', 'VIEW_CV', 'VIDEO_COUNT_x', 'VIEW_P90', 'VIEW_MEAN_V', 'VIDEO_COUNT_y', 'VIRAL_MAGNITUDE', 'MAX_VIRAL_STRENGTH', 'AVG_VIRAL_STRENGTH']


,count,mean,std,min,50%,90%,99%,max
VIDEO_ID,2849.0,7.590238e+18,5.529586e+15,7.561749e+18,7.588114e+18,7.596236e+18,7.613427e+18,7.620368e+18
FOLLOWERS,2849.0,4.324735e+05,1.125925e+06,4.864000e+03,1.503000e+05,9.295800e+05,4.612000e+06,2.120000e+07
FOLLOWING_COUNT,2849.0,3.337992e+02,1.065541e+03,0.000000e+00,6.100000e+01,6.380000e+02,6.638680e+03,1.000000e+04
ENGAGEMENT,2849.0,5.325184e+00,4.815198e+00,1.200000e-01,3.960000e+00,1.101600e+01,1.904640e+01,9.416000e+01
TOTAL_LIKES,2849.0,1.152691e+07,3.106855e+07,1.712000e+03,3.200000e+06,2.450000e+07,1.662880e+08,4.899000e+08
DIGG_COUNT,2849.0,1.672767e+04,3.818599e+04,0.000000e+00,4.729000e+03,4.168000e+04,1.773120e+05,5.543000e+05
VIDEO_COUNT,2849.0,7.193229e+02,1.394927e+03,1.000000e+00,3.740000e+02,1.489200e+03,6.139440e+03,3.590000e+04
COLLAB_SCORE,2809.0,7.840456e+01,5.917600e+00,6.120000e+01,7.720000e+01,8.782000e+01,9.440000e+01,9.880000e+01
VIEW_COUNT,2849.0,3.595335e+05,1.843914e+06,0.000000e+00,4.350000e+04,5.972000e+05,5.452000e+06,5.750000e+07
LIKE_COUNT,2849.0,1.356582e+04,6.977113e+04,0.000000e+00,1.290000e+03,2.312000e+04,2.188360e+05,1.600000e+06


In [9]:

FEATURE_COLS = [
    # Scale & reach
    "FOLLOWERS",  "VIDEO_COUNT", "VIDEO_COUNT_3M_x",

    # View performance
    "VIEW_STD", "VIEW_CV", "VIEW_MEAN_V", 
    # Likes / Comments / Shares / Saves
    "MEAN_LIKES_3M",  "MAX_LIKES_3M",
    "MEAN_COMMENTS_3M",  "MAX_COMMENTS_3M",
    "MEAN_SHARES_3M",  "MAX_SHARES_3M",
    "MEAN_SAVES_3M",  "MAX_SAVES_3M",

    # Engagement rates
    "MEAN_VIEWS_RATE_3M", "MEAN_LIKE_RATE_3M",
    "MEAN_COMMENT_RATE_3M", "MEAN_SHARE_RATE_3M", "MEAN_SAVE_RATE_3M",

    # Posting behavior
    "AVG_POST_GAP", "POSTING_CONSISTENCY", "VIDEOS_PER_WEEK",
    
    # Virality & profile
     "MAX_VIRAL_STRENGTH", "COLLAB_SCORE", 
]

In [6]:

# --- Cleaning ---
df = df[df["VIDEO_COUNT_x"].notna()]          # drop inactive creators (no videos)
df = df[df["CREATOR_ID"] != "nhuhexii"]       # drop anomalous creator

# Targeted NA fills
df["VIEW_STD"]    = df["VIEW_STD"].fillna(0)                          # 1 video → no variance
df["VIEW_CV"]     = df["VIEW_CV"].fillna(0)
df["AVG_POST_GAP"] = df["AVG_POST_GAP"].fillna(df["AVG_POST_GAP"].mean())  # 1 video → use mean


In [7]:
remaining_na = df.isna().sum()
remaining_na = remaining_na[remaining_na > 0]
print("No remaining NAs" if remaining_na.empty else f"Remaining NAs:\n{remaining_na}")

Remaining NAs:
COLLAB_SCORE    40
dtype: int64


In [11]:
# --- Select & validate features ---
df = df[FEATURE_COLS].copy()
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 2845 entries, 0 to 2848
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   FOLLOWERS             2845 non-null   int64  
 1   VIDEO_COUNT           2845 non-null   int64  
 2   VIDEO_COUNT_3M_x      2845 non-null   float64
 3   VIEW_STD              2845 non-null   float64
 4   VIEW_CV               2845 non-null   float64
 5   VIEW_MEAN_V           2845 non-null   float64
 6   MEAN_LIKES_3M         2845 non-null   float64
 7   MAX_LIKES_3M          2845 non-null   float64
 8   MEAN_COMMENTS_3M      2845 non-null   float64
 9   MAX_COMMENTS_3M       2845 non-null   float64
 10  MEAN_SHARES_3M        2845 non-null   float64
 11  MAX_SHARES_3M         2845 non-null   float64
 12  MEAN_SAVES_3M         2845 non-null   float64
 13  MAX_SAVES_3M          2845 non-null   float64
 14  MEAN_VIEWS_RATE_3M    2845 non-null   float64
 15  MEAN_LIKE_RATE_3M     2845

In [12]:
print(df.columns.tolist())

['FOLLOWERS', 'VIDEO_COUNT', 'VIDEO_COUNT_3M_x', 'VIEW_STD', 'VIEW_CV', 'VIEW_MEAN_V', 'MEAN_LIKES_3M', 'MAX_LIKES_3M', 'MEAN_COMMENTS_3M', 'MAX_COMMENTS_3M', 'MEAN_SHARES_3M', 'MAX_SHARES_3M', 'MEAN_SAVES_3M', 'MAX_SAVES_3M', 'MEAN_VIEWS_RATE_3M', 'MEAN_LIKE_RATE_3M', 'MEAN_COMMENT_RATE_3M', 'MEAN_SHARE_RATE_3M', 'MEAN_SAVE_RATE_3M', 'AVG_POST_GAP', 'POSTING_CONSISTENCY', 'VIDEOS_PER_WEEK', 'MAX_VIRAL_STRENGTH', 'COLLAB_SCORE']


In [13]:
df.to_csv("feature_for_clustering.csv", index=False)